# Plant compounds against the target reference sets

For each plant compound, the most similar active inhibitor (pActivity ≥ 6) of p38α, COX-2 and
5-LOX, by Tanimoto similarity on Morgan fingerprints, together with that inhibitor's measured
potency.

The reference molecules are the prepared tables from notebook 3, standardized the same way as
the plant compounds. Stereoisomers are collapsed, compounds with fewer than 25 fingerprint bits
or a molecular weight outside 150–800 are excluded, and every excluded molecule is listed in
the `Excluded` sheet with its reason.

In [ ]:
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

ROOT = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
OUT = ROOT / 'analysis/plants'
OUT.mkdir(parents=True, exist_ok=True)

ACTIVE_CUTOFF = 6.0
HIGHLIGHT = 0.5
MIN_BITS = 25
MW_RANGE = (150, 800)

TARGETS = {'p38a': ('CHEMBL260', ROOT / 'analysis/p38/prepared/p38_prepared.csv'),
           'COX2': ('CHEMBL230', ROOT / 'analysis/cox2/prepared/cox2_prepared.csv'),
           'LOX5': ('CHEMBL215', ROOT / 'analysis/lox5/prepared/lox5_prepared.csv')}

RDLogger.DisableLog('rdApp.*')
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
unch = rdMolStandardize.Uncharger()
print('writing to', OUT.relative_to(ROOT).as_posix())

## Standardization

Same procedure as notebook 3.

In [ ]:
def standardize(smi):
    m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
    if m is None:
        return None
    try:
        m = rdMolStandardize.Cleanup(m)
        if len(Chem.GetMolFrags(m)) > 1:
            m = rdMolStandardize.FragmentParent(m)
        if Chem.GetFormalCharge(m) != 0:
            m = unch.uncharge(m)
        Chem.SanitizeMol(m)
    except Exception:
        return None
    return m

## Reference sets

In [ ]:
refs = {}
for name, (chembl_id, path) in TARGETS.items():
    if not path.exists():
        raise FileNotFoundError(f'{path} not found')
    d = pd.read_csv(path)
    b = np.load(path.parent / 'morgan_bits.npy')
    keep = (d.pActivity >= ACTIVE_CUTOFF).to_numpy()
    refs[name] = {'chembl_id': chembl_id,
                  'table': d.loc[keep].reset_index(drop=True),
                  'bits': b[keep].astype(np.float32)}
    print(f'{name:5s} {chembl_id}: {int(keep.sum()):,} actives of {len(d):,}  '
          f'(median bits_on {np.median(b[keep].sum(1)):.0f})')

## Plant compounds

In [ ]:
plant_files = sorted((ROOT / 'data/coconut').glob('*.csv'))
plants = pd.concat([pd.read_csv(f) for f in plant_files], ignore_index=True)
if 'species' not in plants.columns:
    plants['species'] = None
if 'query_name' in plants.columns:
    plants['species'] = plants['species'].fillna(plants['query_name'])
print(f'{len(plant_files)} plant files, {len(plants):,} rows')

plants['mol'] = [standardize(s) for s in plants.canonical_smiles]
n_bad = int(plants.mol.isna().sum())
plants = plants.loc[plants.mol.notna()].reset_index(drop=True)
plants['flat_smiles'] = [Chem.MolToSmiles(m, isomericSmiles=False) for m in plants.mol]

before = len(plants)
plants = (plants.sort_values(['is_parent', 'citation_count', 'annotation_level'], ascending=False)
          .drop_duplicates(subset=['plant', 'flat_smiles']).reset_index(drop=True))
print(f'{n_bad} failed to standardize; {before:,} -> {len(plants):,} distinct structures per plant '
      f'(stereoisomers collapsed)')

plants['MW'] = [round(Descriptors.MolWt(m), 1) for m in plants.mol]
fp = np.vstack([gen.GetFingerprintAsNumPy(m) for m in plants.mol]).astype(np.float32)
plants['bits_on'] = fp.sum(1).astype(int)
plants['compound_name'] = plants.name.fillna(plants.identifier)

## Size filter

In [ ]:
too_small = plants.bits_on < MIN_BITS
too_light = plants.MW < MW_RANGE[0]
too_heavy = plants.MW > MW_RANGE[1]
reason = np.select(
    [too_small & (too_light | too_heavy), too_small, too_light, too_heavy],
    [f'bits < {MIN_BITS} and MW outside {MW_RANGE}', f'bits_on < {MIN_BITS}',
     f'MW < {MW_RANGE[0]}', f'MW > {MW_RANGE[1]}'], default='')
plants['excluded_reason'] = reason
keep = (reason == '')
print(f'kept {int(keep.sum()):,} of {len(plants):,}')
print(pd.Series(reason[~keep]).value_counts().to_string())

## Tanimoto against each target

A Tanimoto of 1.000 means the plant compound is itself one of the reference molecules
(`exact_match_in_reference`).

In [ ]:
def compare(mask, fps, meta, ref):
    a, b = fps[mask], ref['bits']
    inter = a @ b.T
    sim = inter / np.maximum(a.sum(1)[:, None] + b.sum(1)[None, :] - inter, 1.0)
    best = sim.argmax(axis=1)
    t = ref['table']
    m = meta.loc[mask].reset_index(drop=True)
    return pd.DataFrame({
        'plant': m.plant, 'compound_name': m.compound_name, 'coconut_id': m.identifier,
        'species': m.species,
        'tanimoto': sim.max(axis=1).round(4),
        'exact_match_in_reference': np.where(sim.max(axis=1) >= 0.9999, 'YES', ''),
        'matched_id': t.molecule_chembl_id.to_numpy()[best],
        'matched_pActivity': t.pActivity.to_numpy()[best].round(3),
        'matched_n_measurements': t.n_measurements.to_numpy()[best],
        'plant_MW': m.MW, 'plant_bits_on': m.bits_on,
        'matched_bits_on': ref['bits'].sum(1)[best].astype(int),
        'plant_smiles': m.flat_smiles,
        'matched_smiles': t.std_smiles.to_numpy()[best],
        'citation_count': m.citation_count,
        'excluded_reason': m.excluded_reason,
    }).sort_values('tanimoto', ascending=False).reset_index(drop=True)


results = {name: compare(keep, fp, plants, ref) for name, ref in refs.items()}
excluded = {name: compare(~keep, fp, plants, ref) for name, ref in refs.items()}
for name, r in results.items():
    print(f'{name}: {len(r):,} compounds, max Tanimoto {r.tanimoto.max():.3f}, '
          f'{int((r.tanimoto >= HIGHLIGHT).sum())} at or above {HIGHLIGHT}, '
          f'{int((r.exact_match_in_reference == "YES").sum())} exact matches')

## Summary

In [ ]:
rows = []
for name, r in results.items():
    for plant, g in r.groupby('plant'):
        rows.append({'target': name, 'plant': plant, 'molecules': len(g),
                     'max_tanimoto': g.tanimoto.max(), 'median_tanimoto': round(g.tanimoto.median(), 4),
                     'n_above_0.5': int((g.tanimoto >= .5).sum()),
                     'n_above_0.4': int((g.tanimoto >= .4).sum()),
                     'n_above_0.3': int((g.tanimoto >= .3).sum()),
                     'exact_matches': int((g.exact_match_in_reference == 'YES').sum())})
summary = (pd.DataFrame(rows).pivot(index='plant', columns='target')
           .swaplevel(axis=1).sort_index(axis=1).reset_index())
summary.columns = [f'{b}_{a}' if a else b for a, b in summary.columns]

flat = pd.DataFrame(rows).sort_values(['target', 'max_tanimoto'], ascending=[True, False])
print(flat.to_string(index=False))

above = pd.concat([r.assign(target=n) for n, r in results.items()], ignore_index=True)
above = above[above.tanimoto >= HIGHLIGHT].sort_values('tanimoto', ascending=False)
above = above[['target'] + [c for c in above.columns if c != 'target']].reset_index(drop=True)
print(f'\n{len(above)} compound-target pairs at or above {HIGHLIGHT}')

In [ ]:
print('--- median max-Tanimoto per target: does COX-2 separate from p38a? ---')
for name, r in results.items():
    print(f'  {name:5s} median {r.tanimoto.median():.3f}   mean {r.tanimoto.mean():.3f}   '
          f'max {r.tanimoto.max():.3f}')
delta = results['COX2'].tanimoto.median() - results['p38a'].tanimoto.median()
print(f'\nCOX-2 minus p38a, median: {delta:+.3f}')

## Excel output

In [ ]:
today = dt.date.today().isoformat()
method = pd.DataFrame({'item': [
    *[f'reference: {n}' for n in refs], 'active cutoff', 'fingerprint', 'similarity',
    'standardization', 'stereoisomers', 'size filter', 'plant source', 'COCONUT caveats',
    'exact_match_in_reference', 'generated'],
    'value': [
    *[f"{TARGETS[n][1].relative_to(ROOT).as_posix()}, {len(refs[n]['table']):,} actives "
      f"({TARGETS[n][0]})" for n in refs],
    f'pActivity >= {ACTIVE_CUTOFF} (IC50/Ki at or below 1 uM)',
    'Morgan radius 2, 2048 bits, chirality ignored',
    'Tanimoto on binary fingerprints; best match per plant compound',
    'RDKit Cleanup + FragmentParent + Uncharger + Sanitize, identical to notebook 3, '
    'applied to BOTH sides. The prepared reference files are used, not *_training_raw.csv.',
    'collapsed on flat SMILES: the fingerprint ignores chirality, so stereoisomers are one structure',
    f'kept bits_on >= {MIN_BITS} and {MW_RANGE[0]} <= MW <= {MW_RANGE[1]}; '
    'removed rows are in the Excluded sheet with their reason',
    'COCONUT 2.0 via notebook 6, data/coconut/*.csv',
    'overlapping pages (swept to convergence), taxonomic synonyms (Rosmarinus officinalis '
    'returns 477 molecules, Salvia rosmarinus returns 1), and some mis-annotated organisms',
    'Tanimoto 1.000 means the plant compound IS a reference molecule, already measured '
    'against that target. A confirmation of the data, not a discovery.',
    today]})

xlsx = OUT / 'plant_vs_targets.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    summary.to_excel(w, sheet_name='Summary', index=False)
    above.to_excel(w, sheet_name=f'Above_{HIGHLIGHT}', index=False)
    for name, r in results.items():
        r.to_excel(w, sheet_name=f'All_{name}', index=False)
    pd.concat([e.assign(target=n) for n, e in excluded.items()],
              ignore_index=True).to_excel(w, sheet_name='Excluded', index=False)
    method.to_excel(w, sheet_name='Method', index=False)

for name, r in results.items():
    r.to_csv(OUT / f'plant_vs_{name}.csv', index=False, encoding='utf-8')
print('wrote', xlsx.relative_to(ROOT).as_posix())
for name in results:
    ok = len(results[name]) + len(excluded[name]) == len(plants)
    print(f'rows reconcile ({name}):', ok)

## Known plant compounds in the reference sets

Whether any plant compound with a ChEMBL ID is already among the reference molecules.

In [ ]:
known = {'Ursolic acid': 'CHEMBL169', 'Rosmarinic acid': 'CHEMBL324842',
         'Carnosic acid': 'CHEMBL484853', 'Carnosol': 'CHEMBL218693',
         'Crocetin': 'CHEMBL464792', 'Crocin': 'CHEMBL446785', 'Safranal': 'CHEMBL3183495'}
for name, ref in refs.items():
    present = ref['table'].molecule_chembl_id.isin(known.values())
    hits = ref['table'].loc[present, ['molecule_chembl_id', 'pActivity', 'n_measurements']]
    inv = {v: k for k, v in known.items()}
    hits = hits.assign(compound=[inv[i] for i in hits.molecule_chembl_id])
    print(f'{name}: {len(hits)} of {len(known)} plant compounds are IN the active reference set')
    if len(hits):
        print(hits[['compound', 'molecule_chembl_id', 'pActivity', 'n_measurements']].to_string(index=False))

In [ ]:
ex = above[above.exact_match_in_reference == 'YES']
print(f'{len(ex)} exact matches across both targets -- these are NOT discoveries:')
if len(ex):
    print(ex[['target', 'plant', 'compound_name', 'matched_id', 'matched_pActivity']].to_string(index=False))
print()
real = above[above.exact_match_in_reference != 'YES']
print(f'{len(real)} genuine near-neighbours at or above {HIGHLIGHT}:')
if len(real):
    print(real[['target', 'plant', 'compound_name', 'tanimoto', 'plant_bits_on',
                'matched_id', 'matched_pActivity']].to_string(index=False))

## Notes

- 1.000 is an exact match, not a new hit.
- `matched_pActivity` is the potency of the reference molecule, not a prediction for the plant
  compound.
- Similarity to a known inhibitor suggests activity but does not show it.